# Prior Rollout from Checkpoint

This notebook loads a trained policy checkpoint and performs prior-only rollouts.
Prior rollouts use only the prior network and decoder (no encoder or trajectory observations),
testing whether the model can generate plausible actions from proprioceptive state alone.

**Supports both PPO and distillation checkpoints.**

## Configuration

In [12]:
# === CONFIGURATION ===
# Modify these parameters as needed

# Path to checkpoint directory
CHECKPOINT_PATH = "/home/mila/a/aidan.sirbu/scratch/track-mjx/model_checkpoints/251213_235327_736369"


# Step prefix for loading checkpoints (usually "Network")
STEP_PREFIX = "DistillNetwork"

# Specific step to load (None = latest)
CHECKPOINT_STEP = None

# Number of rollout steps
NUM_STEPS = 200

# Number of rollouts to run
NUM_ROLLOUTS = 4

# Fixed log-variance for prior sampling (lower = less stochastic)
FIXED_LOGVAR = -2.0

# Whether to use deterministic policy (mode vs sample)
DETERMINISTIC = True

# Random seed
SEED = 42

## Imports

In [9]:
%load_ext autoreload
%autoreload 2

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "osmesa"  # Change to "egl" or "glfw" if available

from pathlib import Path

import jax
import jax.numpy as jnp
from jax import random
import mediapy as media
import orbax.checkpoint as ocp
from omegaconf import OmegaConf

from brax.training import distribution
from brax.training.acme import running_statistics, specs

from track_mjx.agent import checkpointing
from track_mjx.agent.mlp_ppo import prior_rollout, intention_network
from track_mjx.analysis import rollout, render, utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load Checkpoint

This section handles loading from both PPO and distillation checkpoints.

In [10]:
def make_abstract_policy_no_critic(cfg, seed: int = 1):
    """
    Create an abstract policy structure without requiring a critic/value network.
    Works for both PPO and distillation checkpoints.
    """
    # Handle backward compatibility for prior_layer_sizes
    prior_layer_sizes = cfg.network_config.get(
        "prior_layer_sizes", cfg.network_config.encoder_layer_sizes
    )
    
    # Set up normalization
    normalize = lambda x, y: x
    if cfg.train_setup.train_config.normalize_observations:
        normalize = running_statistics.normalize
    
    # Create the action distribution
    parametric_action_distribution = distribution.NormalTanhDistribution(
        event_size=cfg.network_config.action_size
    )
    
    # Create just the policy network (no value network)
    policy_network = intention_network.make_intention_policy(
        parametric_action_distribution.param_size,
        latent_size=cfg.network_config.intention_size,
        total_obs_size=cfg.network_config.observation_size,
        reference_obs_size=cfg.network_config.reference_obs_size,
        preprocess_observations_fn=normalize,
        encoder_hidden_layer_sizes=tuple(cfg.network_config.encoder_layer_sizes),
        decoder_hidden_layer_sizes=tuple(cfg.network_config.decoder_layer_sizes),
        prior_hidden_layer_sizes=tuple(prior_layer_sizes),
    )
    
    # Initialize policy params
    key_policy = jax.random.key(seed)
    init_policy_params = policy_network.init(key_policy)
    
    # Create normalizer state
    normalizer_params = running_statistics.init_state(
        specs.Array(cfg.network_config.observation_size, jnp.dtype("float32"))
    )
    
    return (normalizer_params, init_policy_params)


def load_checkpoint_no_critic(checkpoint_path: str, step_prefix: str = "Network", step: int = None):
    """
    Load a checkpoint without requiring a critic/value network.
    Works for both PPO and distillation checkpoints.
    """
    mgr_options = ocp.CheckpointManagerOptions(
        create=False,
        step_prefix=step_prefix,
    )
    ckpt_mgr = ocp.CheckpointManager(checkpoint_path, options=mgr_options)
    
    if step is None:
        step = ckpt_mgr.latest_step()
    
    print(f"Loading checkpoint from {checkpoint_path} at step {step}")
    
    # First load the config
    cfg = OmegaConf.create(
        checkpointing.load_config_from_checkpoint(checkpoint_path, step_prefix, step)
    )
    
    # Create abstract policy without critic
    abstract_policy = make_abstract_policy_no_critic(cfg)
    
    # Load the policy
    policy = ckpt_mgr.restore(
        step,
        args=ocp.args.Composite(
            policy=ocp.args.StandardRestore(abstract_policy),
        ),
    )["policy"]
    
    return {"cfg": cfg, "policy": policy}

In [13]:
ckpt_path = Path(CHECKPOINT_PATH)

# Load checkpoint - this works for both PPO and distillation checkpoints
ckpt = load_checkpoint_no_critic(
    str(ckpt_path), 
    step_prefix=STEP_PREFIX, 
    step=CHECKPOINT_STEP
)

cfg = ckpt["cfg"]
policy_params = ckpt["policy"]

print(f"Loaded checkpoint from: {ckpt_path}")
print(f"Step prefix: {STEP_PREFIX}")
print(f"Network architecture: {cfg.network_config.arch_name}")

Loading checkpoint from /home/mila/a/aidan.sirbu/scratch/track-mjx/model_checkpoints/251213_235327_736369 at step 99
Loaded checkpoint from: /home/mila/a/aidan.sirbu/scratch/track-mjx/model_checkpoints/251213_235327_736369
Step prefix: DistillNetwork
Network architecture: intention


## Create Environment and Prior Policy

In [14]:
# Create environment
env = rollout.create_environment(cfg)

# Extract prior and decoder parameters from the full policy
prior_params, decoder_params, normalizer_params = prior_rollout.extract_prior_decoder_params(policy_params)

# Get network sizes from config
proprioceptive_obs_size = cfg.network_config.observation_size - cfg.network_config.reference_obs_size
intention_latent_size = cfg.network_config.intention_size
action_size = cfg.network_config.action_size

# Handle backward compatibility for prior_layer_sizes
prior_layer_sizes = cfg.network_config.get("prior_layer_sizes", cfg.network_config.encoder_layer_sizes)

print(f"Proprioceptive observation size: {proprioceptive_obs_size}")
print(f"Intention latent size: {intention_latent_size}")
print(f"Action size: {action_size}")

Proprioceptive observation size: 264
Intention latent size: 60
Action size: 38


In [15]:
# Create proprioceptive-only normalizer params (slice to just proprioceptive observations)
proprio_normalizer_params = running_statistics.RunningStatisticsState(
    count=normalizer_params.count,
    mean=normalizer_params.mean[-proprioceptive_obs_size:],
    summed_variance=normalizer_params.summed_variance[-proprioceptive_obs_size:],
    std=normalizer_params.std[-proprioceptive_obs_size:],
)

# Set up normalization function
normalize = lambda x, y: x
if cfg.train_setup.train_config.normalize_observations:
    normalize = running_statistics.normalize

# Create prior policy function
prior_policy_fn = prior_rollout.create_prior_policy(
    prior_network_params=prior_params,
    decoder_network_params=decoder_params,
    normalizer_params=proprio_normalizer_params,
    intention_latent_size=intention_latent_size,
    action_size=action_size,
    proprioceptive_obs_size=proprioceptive_obs_size,
    decoder_hidden_layer_sizes=tuple(cfg.network_config.decoder_layer_sizes),
    prior_hidden_layer_sizes=tuple(prior_layer_sizes),
    preprocess_observations_fn=normalize,
    fixed_logvar=FIXED_LOGVAR,
    deterministic=DETERMINISTIC,
)

print("Prior policy created successfully")

Prior policy created successfully


## Run Prior Rollout

In [18]:
# JIT compile environment functions
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

def run_prior_rollout(rng_key, max_steps=NUM_STEPS):
    """Run a single prior rollout."""
    key_reset, key_rollout = random.split(rng_key)
    
    # Reset environment
    state = jit_reset(key_reset)
    
    # Storage for rollout data
    states = [state]
    actions = []
    extras_list = []

    # Get proprioceptive observations
    if hasattr(state.obs, 'get') or isinstance(state.obs, dict):
        proprio = state.obs.get("proprioception", state.obs)
        if isinstance(proprio, dict):
            from jax import flatten_util
            proprio, _ = flatten_util.ravel_pytree(proprio)
    else:
        proprio = state.obs
    
    key = key_rollout
    for step in range(max_steps):
        key, key_action = random.split(key)
        
        # Get action from prior policy
        action, extras = prior_policy_fn(proprio, key_action)
        
        # Step environment
        state = jit_step(state, action)
        
        states.append(state)
        actions.append(action)
        extras_list.append(extras)
        
        # Check for NaN termination
        if prior_rollout.check_termination_nan(state.data):
            print(f"Rollout terminated early at step {step + 1} due to NaN")
            break
    
    return {
        "states": states,
        "actions": actions,
        "extras": extras_list,
        "num_steps": len(actions),
    }

In [19]:
# Run multiple rollouts
rng = random.PRNGKey(SEED)
rollouts = []

for i in range(NUM_ROLLOUTS):
    rng, rollout_key = random.split(rng)
    print(f"Running rollout {i + 1}/{NUM_ROLLOUTS}...")
    rollout_data = run_prior_rollout(rollout_key, max_steps=NUM_STEPS)
    rollouts.append(rollout_data)
    print(f"  Completed {rollout_data['num_steps']} steps")

# Compute statistics
step_counts = [r["num_steps"] for r in rollouts]
print(f"\nRollout Statistics:")
print(f"  Average steps: {sum(step_counts) / len(step_counts):.1f}")
print(f"  Min steps: {min(step_counts)}")
print(f"  Max steps: {max(step_counts)}")

Running rollout 1/4...
  Completed 200 steps
Running rollout 2/4...
  Completed 200 steps
Running rollout 3/4...
  Completed 200 steps
Running rollout 4/4...
  Completed 200 steps

Rollout Statistics:
  Average steps: 200.0
  Min steps: 200
  Max steps: 200


## Render Rollout

In [28]:
# Render frames
camera_name = "close_profile-ghost"  # Use ghost camera for tracking
frames = env.render(rollouts[3]["states"], camera=camera_name)

In [29]:
# Display video
media.show_video(frames, fps=50)

This browser does not support the video tag.


In [ ]:
# Save video to disk
output_path = ckpt_path / "prior_rollout.mp4"
media.write_video(output_path, frames, fps=realtime_framerate)
print(f"Video saved to: {output_path}")

## Analyze Prior Latent Distributions

In [ ]:
import matplotlib.pyplot as plt

# Extract prior means and intentions from the best rollout
prior_means = jnp.stack([e["prior_mean"] for e in best_rollout["extras"]])
intentions = jnp.stack([e["intention"] for e in best_rollout["extras"]])

print(f"Prior means shape: {prior_means.shape}")
print(f"Intentions shape: {intentions.shape}")

# Plot first few latent dimensions over time
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

num_dims_to_plot = min(5, intention_latent_size)
timesteps = range(len(prior_means))

axes[0].set_title("Prior Mean (first 5 dimensions)")
for i in range(num_dims_to_plot):
    axes[0].plot(timesteps, prior_means[:, i], label=f"dim {i}")
axes[0].legend()
axes[0].set_xlabel("Timestep")
axes[0].set_ylabel("Value")

axes[1].set_title("Sampled Intention (first 5 dimensions)")
for i in range(num_dims_to_plot):
    axes[1].plot(timesteps, intentions[:, i], label=f"dim {i}")
axes[1].legend()
axes[1].set_xlabel("Timestep")
axes[1].set_ylabel("Value")

plt.tight_layout()
plt.show()

## Save Rollout Data

In [ ]:
# Save the best rollout data to HDF5
save_data = {
    "actions": jnp.stack(best_rollout["actions"]),
    "prior_means": prior_means,
    "intentions": intentions,
    "num_steps": best_rollout["num_steps"],
}

save_path = ckpt_path / "prior_rollout_data.h5"
utils.save_to_h5py(str(save_path), save_data)
print(f"Rollout data saved to: {save_path}")